### Block 0 — Install packages

In [4]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn tqdm pandas huggingface_hub
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.0 MB/s eta 0:00:00


In [1]:
from huggingface_hub import login

login()

### Block 1 — Imports and Gemini API client

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import os


MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading {MODEL_ID} in 4-bit quantization...")


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token_id = tokenizer.eos_token_id


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=10,
    return_full_text=False,
    temperature=0.01,
    do_sample=False
)

print("Model loaded successfully!")

Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit quantization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully!


### Block 2 — Load FIQA dataset

In [3]:


dataset = load_dataset("TheFinAI/flare-fiqasa", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])

README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 3 — Labels and normalization

In [4]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map the raw model output to one of: 'negative', 'neutral', 'positive'.
    """
    text = (raw or "").strip().lower()

    # Exact match first
    for label in LABELS:
        if label in text:
            return label

    # Handle short variants like "pos" or "neg"
    if "pos" in text:
        return "positive"
    if "neg" in text:
        return "negative"

    # Fallback
    return "neutral"


### Block 4 — Llama sentiment classifier

In [5]:
def predict_label(text: str) -> str:
    """
    Call Llama 3.1 (Local Pipeline) and return a normalized label.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert financial sentiment classifier. "
                "Classify the sentiment of the financial news snippet as exactly one of: negative, neutral, or positive. "
                "Return only one word."
            )
        },
        {
            "role": "user",
            "content": f"Text: {text}\nAnswer:"
        }
    ]

    # 调用模型生成
    # pipeline 会自动应用 Llama 3 的 chat template (<|begin_of_text|>...)
    outputs = text_generator(
        messages,
        pad_token_id=tokenizer.eos_token_id
    )

    # 提取生成的文本
    # outputs[0]["generated_text"] 对于 chat pipeline 通常直接返回内容
    raw = outputs[0]["generated_text"]

    # 清洗并返回
    return normalize_prediction(raw)

### Block 5 — Evaluation loop over FIQA test split

In [6]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    text = example["text"]
    true_label = example["answer"].strip().lower()

    y_true.append(true_label)

    pred_label = predict_label(text)
    y_pred.append(pred_label)


100%|██████████| 235/235 [00:46<00:00,  5.11it/s]


### Block 6 — Compute metrics and inspect predictions

In [7]:
# Overall accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Build DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Accuracy: 0.7745

Classification report:
              precision    recall  f1-score   support

    negative     0.8356    0.8026    0.8188        76
     neutral     0.2105    0.4444    0.2857        18
    positive     0.9113    0.8014    0.8528       141

    accuracy                         0.7745       235
   macro avg     0.6525    0.6828    0.6524       235
weighted avg     0.8331    0.7745    0.7984       235


First 10 predictions:
                                                                                                                            text     gold     pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative negative
                                                                         Legal & General share price: Finance chief to step down negative negative
                                                                 Kingfisher share price slides on cost to implemen